# CoAP: REST untuk perangkat terbatas

GET, PUT, Observe, dan Block-wise lewat UDP — terhadap node rumah kaca simulasi di jaringan memori yang kehilangan paketnya bisa Anda atur.

## Persiapan
> Bekerja dari hasil clone? Jalankan `dotnet pack -c Release -o artifacts/packages` di root repo lalu tambahkan
> `#i "nuget: <repo>/artifacts/packages"` sebelum baris `#r`.

In [ ]:
#r "nuget: IoTCom.Net, 0.8.0-preview.1"
#r "nuget: IoTCom.Net.Native.Modbus, 0.8.0-preview.1"

## Satu pesan di jalur
Contoh RFC 7252: CON GET /temperature, 16 byte.

In [ ]:
using IoTCom.Net.Protocols.Coap;

var get = new CoapMessage { Code = CoapCode.Get, MessageId = 0x7D34 };
get.UriPath = "/temperature";
var wire = get.Encode();
Console.WriteLine(Convert.ToHexString(wire));
foreach (var f in CoapAnatomy.Describe(wire)) Console.WriteLine($"{f.Name,-10} {f.Value}");

## Perangkat dan client

In [ ]:
using System.Net;
using IoTCom.Net.Transports;

var net = new InMemoryDatagramNetwork();
var node = new IPEndPoint(IPAddress.Parse("10.0.0.40"), 5683);
await using var server = CoapServer.Create(o => o.UseInMemory(net, node));
await using var device = new CoapDeviceSimulator(server);
await server.StartAsync();

await using var coap = CoapClient.Create(o => o.UseInMemory(net).UseServer(node));
await coap.ConnectAsync();
foreach (var link in await coap.DiscoverAsync()) Console.WriteLine($"{link.Path,-22} {link.ResourceType} {(link.Observable ? "(observable)" : "")}");
Console.WriteLine((await coap.GetAsync("/sensors/temperature")).PayloadText);
Console.WriteLine((await coap.GetAsync("/sensors/temperature", CoapContentFormat.SenMLJson)).PayloadText);

## Observe
Perangkat mendorong notifikasi saat nilainya berubah.

In [ ]:
using var cts = new CancellationTokenSource();
var count = 0;
var watch = Task.Run(async () => { await foreach (var n in coap.ObserveAsync("/sensors/soil", ct: cts.Token)) { Console.WriteLine($"soil {n.PayloadText} (obs {n.ObserveSequence})"); if (++count == 3) break; } });
await coap.PutAsync("/actuators/valve", "80");
while (!watch.IsCompleted) await device.StepAsync();

## Jaringan yang kehilangan paket
30 % datagram hilang; pesan confirmable dikirim ulang sampai di-ACK.

In [ ]:
net.LossRate = 0.3;
coap.Options.Transmission.AckTimeout = TimeSpan.FromMilliseconds(50);
coap.Options.Transmission.MaxRetransmit = 8;
var log = await coap.GetAsync("/device/log");   // Block2: several exchanges
Console.WriteLine($"{log.Payload.Length} bytes; {coap.Statistics.RetransmissionCount} retransmissions; {net.Dropped} datagrams lost");

## Lebih lanjut
Demo Galeri *Rumah kaca pintar lewat CoAP* menggambar setiap datagram di diagram urutan pesan. Lihat `docs/id/protocols/coap.md`.

*IoTCom.Net — dibuat oleh Gravicode Studios dipimpin oleh Kang Fadhil.*